In [3]:
from pathlib import Path
from time import perf_counter

import numpy as np
import pandas as pd
from imblearn.over_sampling import RandomOverSampler
from imblearn.under_sampling import RandomUnderSampler
from lightgbm import LGBMClassifier
from sklearn.metrics import (
    accuracy_score, confusion_matrix, f1_score, precision_score, recall_score
)
from sklearn.model_selection import StratifiedKFold, train_test_split

SEED = 42

from pathlib import Path
import pandas as pd

SEED = 42

# 현재 VS Code 작업 폴더 경로
ROOT = Path.cwd()

# CSV 파일 찾기
candidates = [
    ROOT / "nepal_building_damage.csv",
    ROOT / "nepal_building_damage (1).csv"
]

filepath = next(
    (p for p in candidates if p.exists()),
    None
)

if filepath is None:
    raise FileNotFoundError(
        f"CSV 파일을 찾을 수 없습니다. 현재 폴더: {ROOT}"
    )

# 원본 데이터 불러오기
df_original = pd.read_csv(filepath)

# 원본 보존하고 분석용 복사본 생성
df_eda = df_original.copy(deep=True)

print("CSV 경로:", filepath)
print("원본 데이터:", df_original.shape)
print("분석용 복사본:", df_eda.shape)

if filepath is None:
    raise FileNotFoundError(f"CSV 파일을 {ROOT} 폴더에 배치해 주세요: {[p.name for p in candidates]}")

df_original = pd.read_csv(filepath)
required = {"building_id", "damage_grade", "geo_level_1_id", "geo_level_2_id", "geo_level_3_id"}
if missing := required.difference(df_original.columns):
    raise KeyError(f"CSV 필수 열이 없습니다: {sorted(missing)}")

# 기존 38개 모델 입력은 모두 유지. ID와 정답은 제외.
X_base = df_original.drop(columns=["building_id", "damage_grade"]).copy()
y = df_original["damage_grade"].copy()
assert set(y.unique()) == {1, 2, 3}, "피해등급 값 1·2·3을 확인하세요."

# 사전 탐색에서 범주형 변수 쌍의 보정 Cramer's V가 높은 조합 5개
pairs = {
    "geo1_x_mudstone": ("geo_level_1_id", "has_superstructure_mud_mortar_stone"),
    "foundation_x_mudstone": ("foundation_type", "has_superstructure_mud_mortar_stone"),
    "roof_x_rceng": ("roof_type", "has_superstructure_rc_engineered"),
    "groundfloor_x_cementbrick": ("ground_floor_type", "has_superstructure_cement_mortar_brick"),
    "otherfloor_x_mudstone": ("other_floor_type", "has_superstructure_mud_mortar_stone"),
}
X_joint = X_base.copy()
for name, (col1, col2) in pairs.items():
    X_joint[name] = X_joint[col1].astype(str) + "|" + X_joint[col2].astype(str)

# 이전 사용자가 요청한 대·중·소 통합지역의 추가 기여도도 별도 비교
X_geo = X_joint.copy()
X_geo["region_combined"] = (
    X_geo["geo_level_1_id"].astype(str) + "_" +
    X_geo["geo_level_2_id"].astype(str) + "_" +
    X_geo["geo_level_3_id"].astype(str)
)
feature_sets = {
    "원본38": X_base,
    "원본+결합5": X_joint,
    "원본+통합지역": X_base.assign(region_combined=X_geo["region_combined"]),
    "원본+결합5+통합지역": X_geo,
}

# 60% train / 20% validation / 20% test
idx = np.arange(len(df_original))
trainval_idx, test_idx = train_test_split(
    idx, test_size=0.20, stratify=y, random_state=SEED
)
train_idx, val_idx = train_test_split(
    trainval_idx, test_size=0.25, stratify=y.iloc[trainval_idx], random_state=SEED
)
print('전체/Train/Validation/Test:',len(idx),len(train_idx),len(val_idx),len(test_idx))
print('Train 등급별 건수:', y.iloc[train_idx].value_counts().sort_index().to_dict())

geo_cols = {"geo_level_1_id", "geo_level_2_id", "geo_level_3_id"}


from pandas.api.types import is_numeric_dtype

geo_cols = {
    "geo_level_1_id",
    "geo_level_2_id",
    "geo_level_3_id"
}

def encode_per_train(data, train_ids, valid_ids):

    # 1. 학습용과 검증용 데이터 복사
    train = data.iloc[train_ids].copy()
    valid = data.iloc[valid_ids].copy()

    # 2. 범주형으로 처리할 컬럼 찾기
    categorical = [
        c for c in data.columns
        if c in geo_cols
        or not is_numeric_dtype(data[c])
    ]

    # 3. 범주형 자료형으로 변환
    for c in categorical:

        train[c] = train[c].astype("category")

        # 검증 데이터도 학습 데이터의 범주 기준 사용
        valid[c] = pd.Categorical(
            valid[c],
            categories=train[c].cat.categories
        )

    return train, valid, categorical


def resample_train(X, y_values, method):
    """훈련 데이터만 리샘플링; 검증과 테스트는 원래 분포 유지."""
    if method in {"none", "weight_balanced"}:
        return X, y_values
    if method == "under_full":
        sampler = RandomUnderSampler(random_state=SEED)
    elif method in {"under_partial", "under_partial+weight_balanced"}:
        # 다수인 2등급을 3등급 수까지만 축소, 1등급과 3등급은 유지.
        cap = int(y_values.value_counts().loc[3])
        sampler = RandomUnderSampler(sampling_strategy={2: cap}, random_state=SEED)
    elif method == "over_full":
        sampler = RandomOverSampler(random_state=SEED)
    else:
        raise ValueError(f"알 수 없는 샘플링 방식: {method}")
    return sampler.fit_resample(X, y_values)

def evaluate(X, train_ids, valid_ids, feature_name, method):
    start = perf_counter()
    tr, va, cat = encode_per_train(X, train_ids, valid_ids)
    yt, yv = y.iloc[train_ids].copy(), y.iloc[valid_ids].to_numpy()
    tr, yt = resample_train(tr, yt, method)
    # LightGBM sklearn 기본 트리 수=100. 아래 변수는 실행 재현성을 위한 설정.
    model = LGBMClassifier(
        random_state=SEED,
        n_estimators=100,
        n_jobs=4,
        verbosity=-1,
        class_weight="balanced" if "weight_balanced" in method else None,
    )
    model.fit(tr, yt, categorical_feature=cat)
    pred = model.predict(va)
    cm = confusion_matrix(yv, pred, labels=[1,2,3])
    n3 = int(cm[2].sum())
    return {
        "features": feature_name,
        "sampling": method,
        "train_samples": len(yt),
        "macro_f1": f1_score(yv, pred, average="macro"),
        "accuracy": accuracy_score(yv, pred),
        "recall1": recall_score(yv, pred, labels=[1], average=None)[0],
        "recall2": recall_score(yv, pred, labels=[2], average=None)[0],
        "recall3": cm[2,2] / n3,
        "precision3": precision_score(yv, pred, labels=[3], average=None, zero_division=0)[0],
        "err3to1": cm[2,0] / n3,
        "err3to2": cm[2,1] / n3,
        "err3to1_count": int(cm[2,0]),
        "elapsed_seconds": round(perf_counter()-start,1),
    }

sampling_methods = [
    "none", "weight_balanced", "under_partial", "under_full",
    "over_full", "under_partial+weight_balanced"
]

# 1) 기본 피처/파생 5개 및 6가지 샘플링 비교
rows=[]
for feat_name in ["원본38", "원본+결합5"]:
    for method in sampling_methods:
        r = evaluate(feature_sets[feat_name], train_idx, val_idx, feat_name, method)
        rows.append(r)
        print('검증', feat_name, method, 'Macro F1', round(r['macro_f1'], 5), flush=True)
        pd.DataFrame(rows).to_csv(ROOT / 'nepal_model_validation_comparison.csv', index=False, encoding='utf-8-sig')

# 2) 통합지역 추가 여부를 두 주요 방법에서 비교
for feat_name in ["원본+통합지역", "원본+결합5+통합지역"]:
    for method in ["none", "under_partial"]:
        r = evaluate(feature_sets[feat_name], train_idx, val_idx, feat_name, method)
        rows.append(r)
        pd.DataFrame(rows).to_csv(ROOT / 'nepal_model_validation_comparison.csv', index=False, encoding='utf-8-sig')

# 3) 각 결합 피처 1개씩 추가하는 제거/추가 효과 실험
ablation=[]
for name in pairs:
    one = X_base.assign(**{name: X_joint[name]})
    for method in ["none", "under_partial"]:
        r = evaluate(one, train_idx, val_idx, '원본+'+name, method)
        ablation.append(r)
        pd.DataFrame(ablation).to_csv(ROOT / 'nepal_single_feature_ablation.csv', index=False, encoding='utf-8-sig')

# 4) test는 그대로 보류한 채 train+validation에서 3-Fold 반복 검증
folds = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)
cv_results=[]
for feat_name in ["원본38", "원본+결합5"]:
    for method in ["none", "under_partial"]:
        for fold, (tr_rel, va_rel) in enumerate(
            folds.split(np.zeros(len(trainval_idx)), y.iloc[trainval_idx]), 1
        ):
            r = evaluate(feature_sets[feat_name], trainval_idx[tr_rel], trainval_idx[va_rel], feat_name, method)
            r['fold'] = fold
            cv_results.append(r)
            print('3-Fold',feat_name,method,fold,round(r['macro_f1'],5),flush=True)
            pd.DataFrame(cv_results).to_csv(ROOT / 'nepal_model_cv_comparison.csv', index=False, encoding='utf-8-sig')
print('\n완료: 결과 CSV 3개를 생성했습니다. Test는 모델 선택에 사용하지 않았습니다.')


CSV 경로: c:\t_project_1_1\nepal_building_damage.csv
원본 데이터: (260601, 40)
분석용 복사본: (260601, 40)
전체/Train/Validation/Test: 260601 156360 52120 52121
Train 등급별 건수: {1: 15074, 2: 88955, 3: 52331}


C:\Users\jhj31\AppData\Local\Temp\ipykernel_29908\577905515.py:130: Pandas4Warning: Constructing a Categorical with a dtype and values containing non-null entries not in that dtype's categories is deprecated and will raise in a future version.
  valid[c] = pd.Categorical(
C:\Users\jhj31\AppData\Local\Temp\ipykernel_29908\577905515.py:130: Pandas4Warning: Constructing a Categorical with a dtype and values containing non-null entries not in that dtype's categories is deprecated and will raise in a future version.
  valid[c] = pd.Categorical(


검증 원본38 none Macro F1 0.69503


C:\Users\jhj31\AppData\Local\Temp\ipykernel_29908\577905515.py:130: Pandas4Warning: Constructing a Categorical with a dtype and values containing non-null entries not in that dtype's categories is deprecated and will raise in a future version.
  valid[c] = pd.Categorical(
C:\Users\jhj31\AppData\Local\Temp\ipykernel_29908\577905515.py:130: Pandas4Warning: Constructing a Categorical with a dtype and values containing non-null entries not in that dtype's categories is deprecated and will raise in a future version.
  valid[c] = pd.Categorical(


검증 원본38 weight_balanced Macro F1 0.67682


...

3-Fold 원본+결합5 under_partial 1 0.69444


C:\Users\jhj31\AppData\Local\Temp\ipykernel_29908\577905515.py:130: Pandas4Warning: Constructing a Categorical with a dtype and values containing non-null entries not in that dtype's categories is deprecated and will raise in a future version.
  valid[c] = pd.Categorical(
C:\Users\jhj31\AppData\Local\Temp\ipykernel_29908\577905515.py:130: Pandas4Warning: Constructing a Categorical with a dtype and values containing non-null entries not in that dtype's categories is deprecated and will raise in a future version.
  valid[c] = pd.Categorical(


3-Fold 원본+결합5 under_partial 2 0.69406


C:\Users\jhj31\AppData\Local\Temp\ipykernel_29908\577905515.py:130: Pandas4Warning: Constructing a Categorical with a dtype and values containing non-null entries not in that dtype's categories is deprecated and will raise in a future version.
  valid[c] = pd.Categorical(
C:\Users\jhj31\AppData\Local\Temp\ipykernel_29908\577905515.py:130: Pandas4Warning: Constructing a Categorical with a dtype and values containing non-null entries not in that dtype's categories is deprecated and will raise in a future version.
  valid[c] = pd.Categorical(


3-Fold 원본+결합5 under_partial 3 0.69474

완료: 결과 CSV 3개를 생성했습니다. Test는 모델 선택에 사용하지 않았습니다.
